# 06 — Hybrid Approach: K-Means Clustering + Logistic Regression

Objectif : Approche hybride avec évaluation spécifique du clustering (Silhouette) et de la classification (Accuracy/AUC).

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"

try:
    if 'spark' in locals(): spark.stop()
except: pass

spark = (
    SparkSession.builder
    .appName("AmazonFoodReviews-KMeans-Performance")
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem")
    .getOrCreate()
)

DATA_PATH = Path("../data/Reviews.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../Reviews.csv")

In [ ]:
# Preprocessing 20% sample
df = spark.read.csv(str(DATA_PATH), header=True, inferSchema=True, multiLine=True, escape='"') \
          .select("Score", "Text").na.drop() \
          .sample(0.2, seed=42) \
          .withColumn("Text", F.lower(F.regexp_replace(F.col("Text"), r"[^a-zA-Z\s]", "")))

tokenizer = RegexTokenizer(inputCol="Text", outputCol="words", pattern="\\W")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashing = HashingTF(inputCol="filtered", outputCol="raw", numFeatures=5000)
idf = IDF(inputCol="raw", outputCol="features")

pipeline = Pipeline(stages=[tokenizer, remover, hashing, idf])
df_features = pipeline.fit(df).transform(df).cache()

## 1. K-Means Performance Metrics

Comment évaluer un modèle sans labels (unsupervised) ?

In [ ]:
kmeans = KMeans(featuresCol="features", k=3, seed=42, maxIter=15)
model = kmeans.fit(df_features)
predictions = model.transform(df_features)

# 1. Silhouette Score (Mesure de la séparation des clusters)
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette Score: {silhouette:.4f}")
print("(Note: Un score proche de 1 signifie que les clusters sont très bien séparés.)")

# 2. WSSSE (Within Set Sum of Squared Errors)
wssse = model.getSummary().trainingCost
print(f"WSSSE (Inertie): {wssse:.2f}")

## 2. Hybrid Classification Benchmark (LR)

On évalue maintenant si la LR peut prédire ces clusters.

In [ ]:
train, test = predictions.randomSplit([0.8, 0.2], seed=42)
lr = LogisticRegression(featuresCol="features", labelCol="prediction", maxIter=15)
lr_results = lr.fit(train).transform(test)

acc_eval = MulticlassClassificationEvaluator(labelCol="prediction", predictionCol="prediction_lr")
accuracy = MulticlassClassificationEvaluator(labelCol="prediction", metricName="accuracy").evaluate(lr_results)
print(f"LR Accuracy on Clusters: {accuracy:.4f}")